# Lookup & Sorting Gaps

## Age

In [1]:
import pandas as pd

In [2]:
lookup = pd.read_csv('data/LAD11_LAD21.csv')
age_11 = pd.read_csv('data/Age_2011.csv') 
age_21 = pd.read_csv('data/Age_2021.csv')

In [3]:
age_11_mapped = pd.merge(age_11, lookup[['LAD11CD', 'LAD21CD']], on='LAD11CD', how='left')

In [4]:
age_11_aggr = age_11_mapped.groupby('LAD21CD')['YAge11_Pct'].mean().reset_index()

In [5]:
age_df= pd.merge(age_11_aggr, age_21[['LAD21CD', 'YAge_Pct_21']], on='LAD21CD')

In [6]:
age_df['YAge_Pct_Change'] = ((age_df['YAge_Pct_21'] - age_df['YAge11_Pct']) / age_df['YAge11_Pct']) * 100

In [7]:
age_df.to_csv('Age_Pct.csv', index=False)
print(age_df.head())

     LAD21CD  YAge11_Pct  YAge_Pct_21  YAge_Pct_Change
0  E06000001        12.0          9.8       -18.333333
1  E06000002        14.4         11.8       -18.055556
2  E06000003        11.3          8.9       -21.238938
3  E06000004        12.1          9.1       -24.793388
4  E06000005        10.6          9.2       -13.207547


## Health

In [8]:
health_11 = pd.read_csv('data/Health_2011.csv') 
health_21 = pd.read_csv('data/Health_2021.csv')

In [9]:
#combine bad and very bad health into a single column
#simplifies the process
health_11['PHealth_Pct_11'] = health_11['BHPct_11'] + health_11['VBHPct_11']
health_21['PHealth_Pct_21'] = health_21['BH_Pct21'] + health_21['VBH_Pct21']

In [10]:
health_11_mapped = pd.merge(health_11, lookup[['LAD11CD', 'LAD21CD']], on='LAD11CD', how='left')

In [11]:
health_11_aggr = health_11_mapped.groupby('LAD21CD')['PHealth_Pct_11'].mean().reset_index()

In [12]:
health_df = pd.merge(health_11_aggr, health_21[['LAD21CD', 'PHealth_Pct_21']], on='LAD21CD')

In [13]:
health_df['Health_Pct_Change'] = (
    (health_df['PHealth_Pct_21'] - health_df['PHealth_Pct_11']) 
    / health_df['PHealth_Pct_11']
) * 100

In [14]:
health_df.to_csv('Health_Pct.csv', index=False)
print(health_df.head())

     LAD21CD  PHealth_Pct_11  PHealth_Pct_21  Health_Pct_Change
0  E06000001            8.14             8.0          -1.719902
1  E06000002            7.62             7.3          -4.199475
2  E06000003            7.84             7.6          -3.061224
3  E06000004            6.33             6.3          -0.473934
4  E06000005            5.87             5.9           0.511073


## Migration

In [15]:
import re

In [16]:
mig_11 = pd.read_csv('data/Mig_2011.csv') 
mig_21 = pd.read_csv('data/Mig_2021.csv')

In [17]:
#discrepancies in rows of data, this is to clean it
def clean_area_names(text):
    if pd.isna(text): return ""
    text = str(text).lower()
# remove punctuation and extra whitespace
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

In [18]:
mig_11['AddMig_11'] = mig_11['TotRes_ME11'] + mig_11['TotRes_SA11']

In [19]:
mig_11['name_clean'] = mig_11['LAD11NM'].apply(clean_area_names)
lookup['name_clean'] = lookup['LAD11NM'].apply(clean_area_names)

In [20]:
standard_mig11 = pd.merge(mig_11[['name_clean', 'TotRes_ME11', 'TotRes_SA11', 'Tot_Mig11', 'AddMig_11']], 
    lookup[['name_clean', 'LAD21CD']].drop_duplicates(), 
    on='name_clean', 
    how='inner'
)
standard_mig11.head()

,name_clean,TotRes_ME11,TotRes_SA11,Tot_Mig11,AddMig_11,LAD21CD
0,adur,0,9,85,9,E07000223
1,allerdale,4,0,93,4,E07000026
2,amber valley,0,5,33,5,E07000032
3,arun,5,10,164,15,E07000224
4,ashfield,0,15,41,15,E07000170


In [21]:
actual_mig11= standard_mig11.groupby('LAD21CD').agg({
    'TotRes_ME11': 'sum',
    'TotRes_SA11': 'sum',
    'AddMig_11': 'sum',
    'Tot_Mig11': 'sum'
}).reset_index()

actual_mig11.head()

,LAD21CD,TotRes_ME11,TotRes_SA11,AddMig_11,Tot_Mig11
0,E06000001,0,1,1,36
1,E06000002,76,64,140,706
2,E06000003,3,0,3,17
3,E06000004,26,30,56,164
4,E06000005,0,21,21,86


In [22]:
mig_21['AddMig_21'] = mig_21['TotRes_ME21'] + mig_21['TotRes_SA21']

In [23]:
actual_mig21 = pd.merge(mig_21[['LAD21CD', 'TotRes_ME21', 'TotRes_SA21', 'AddMig_21', 'TotMig_21']], 
    lookup[['LAD21CD']].drop_duplicates(), 
    on='LAD21CD', 
    how='inner'
)
actual_mig21.head()

,LAD21CD,TotRes_ME21,TotRes_SA21,AddMig_21,TotMig_21
0,E07000223,2,2,4,50
1,E07000026,1,0,1,19
2,E07000032,0,2,2,33
3,E07000224,1,6,7,132
4,E07000170,1,11,12,54


In [24]:
actual_mig = pd.merge(actual_mig11, actual_mig21, on='LAD21CD', how='inner')
actual_mig.head()

,LAD21CD,TotRes_ME11,TotRes_SA11,AddMig_11,Tot_Mig11,TotRes_ME21,TotRes_SA21,AddMig_21,TotMig_21
0,E06000001,0,1,1,36,12,1,13,26
1,E06000002,76,64,140,706,16,105,121,290
2,E06000003,3,0,3,17,2,3,5,33
3,E06000004,26,30,56,164,27,20,47,181
4,E06000005,0,21,21,86,4,15,19,70


In [25]:
actual_mig['MigProp_Pct11'] = (actual_mig['AddMig_11'] / actual_mig['Tot_Mig11']) * 100
actual_mig['MigProp_Pct21'] = (actual_mig['AddMig_21'] / actual_mig['TotMig_21']) * 100

In [26]:
actual_mig['Mig_Change'] = actual_mig['MigProp_Pct21'] - actual_mig['MigProp_Pct11']

In [27]:
print(f"Final standardized rows: {len(actual_mig)}")
print(actual_mig[['LAD21CD', 'MigProp_Pct11', 'MigProp_Pct21', 'Mig_Change']].head())

Final standardized rows: 327
     LAD21CD  MigProp_Pct11  MigProp_Pct21  Mig_Change
0  E06000001       2.777778      50.000000   47.222222
1  E06000002      19.830028      41.724138   21.894110
2  E06000003      17.647059      15.151515   -2.495544
3  E06000004      34.146341      25.966851   -8.179491
4  E06000005      24.418605      27.142857    2.724252


In [28]:
actual_mig.to_csv('Mig_Pct.csv', index=False)

## Violent Crime

In [29]:
PFAlookup = pd.read_csv('data/LAD21_PFA21.csv')
crime_pfa = pd.read_csv('data/PFA_Crime.csv')

In [30]:
#clean columns 
PFAlookup['LAD21CD'] = PFAlookup['LAD21CD'].astype(str).str.strip()
PFAlookup['PFA21CD'] = PFAlookup['PFA21CD'].astype(str).str.strip()
crime_pfa['PFA21CD'] = crime_pfa['PFA21CD'].astype(str).str.strip()

In [31]:
crime_lad = pd.merge(
    PFAlookup[['LAD21CD', 'PFA21CD']].drop_duplicates(), 
    crime_pfa[['PFA21CD', 'Pct_VATP_11_21']], 
    on='PFA21CD', 
    how='inner'
)
crime_lad.head()

,LAD21CD,PFA21CD,Pct_VATP_11_21
0,E09000002,E23000001,21.8
1,E09000003,E23000001,21.8
2,E09000004,E23000001,21.8
3,E09000005,E23000001,21.8
4,E09000006,E23000001,21.8


In [32]:
crime_lad.to_csv('Crime_Pct.csv', index=False)

## Sorting Gaps

In [34]:
### there were some missing data gaps, upon a rough inspection it was due to different organisations of the channels or counties
### i'm not sure why that is when the LAD's were the same

In [47]:
import geopandas as gpd

In [48]:
shapefile = gpd.read_file('data/shapefile/LAD_MAY_2021_UK_BFC.shp')
sort= shapefile[shapefile['LAD21CD'].str.startswith(('E', 'W'))].copy()

In [49]:
health = pd.read_csv('aggregated data/Health_Pct.csv')
age = pd.read_csv('aggregated data/Age_Pct.csv')
crime = pd.read_csv('aggregated data/Crime_Pct.csv')
mig = pd.read_csv('aggregated data/Mig_Pct.csv')

In [50]:
datasets = [health, age, crime, mig]
for df in datasets:
    df['LAD21CD'] = df['LAD21CD'].astype(str).str.strip()
    # remove any rows that aren't actually LAD codes (like national totals)
    df = df[df['LAD21CD'].str.startswith(('E', 'W'))]

In [51]:
#adds the following variables onto the new gdf with health and the shapefile
master_gdf = shapefile.merge(health[['LAD21CD', 'Health_Pct_Change']], on='LAD21CD', how='left')
master_gdf = master_gdf.merge(age[['LAD21CD', 'YAge_Pct_Change']], on='LAD21CD', how='left')
master_gdf = master_gdf.merge(crime[['LAD21CD', 'Pct_VATP_11_21']], on='LAD21CD', how='left')
master_gdf = master_gdf.merge(mig[['LAD21CD', 'Mig_Change']], on='LAD21CD', how='left')

In [52]:
master_gdf = master_gdf[master_gdf['LAD21CD'].str.startswith(('E', 'W'))]

In [53]:
master_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 331 entries, 0 to 373
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   LAD21CD            331 non-null    object  
 1   LAD21NM            331 non-null    object  
 2   LAD21NMW           22 non-null     object  
 3   BNG_E              331 non-null    int64   
 4   BNG_N              331 non-null    int64   
 5   LONG               331 non-null    float64 
 6   LAT                331 non-null    float64 
 7   GlobalID           331 non-null    object  
 8   geometry           331 non-null    geometry
 9   Health_Pct_Change  308 non-null    float64 
 10  YAge_Pct_Change    308 non-null    float64 
 11  Pct_VATP_11_21     331 non-null    float64 
 12  Mig_Change         327 non-null    float64 
dtypes: float64(6), geometry(1), int64(2), object(4)
memory usage: 36.2+ KB


In [54]:
master_gdf.to_file("master_file.geojson", driver='GeoJSON')
master_gdf.drop(columns='geometry').to_csv("master_file.csv", index=False)